In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
"""WEDNESDAY"""
# unit and sanity checks!!!
# ---------*
# # cases
# 1. regular, no filtering
# 2. only mb or mf trials (balance and no balance)
# 3. both mb and mf trials, balanced
# 4. use passed idx_subsamps
# ---------*

# rerun strategy balancing analyses, epochs
# add movement (average normalized me on a trial)
"""--------------------------------------------"""
# add time
# one regressor

## both

In [ ]:
"""TODO: inform michael"""
# no outlier trial filtering at the moment
# for session subsampling, i divided the number of tents so that the frequency of slow drift is the same

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id, num_tents=12)
encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
encoder.verify(subtract_baseline=False)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(subj_id, sess_id)
se.plot_cvr2()
se.plot_dr2()

# strategy split

In [ ]:
# TODO
# 0. clean up plotting fn
# 1. aggregate across sessions

In [ ]:
encoder.trial_data.rename({"current_block_side": "block_side"})

In [ ]:
encoder.dm_names

In [ ]:
encoder = Encoder(subj_id, sess_id, num_tents=5, separate_drift=False)
encoder.fit_encoder()
encoder.encoder_predict()
encoder.verify(subtract_baseline=True)

In [ ]:
encoder_mb = Encoder(subj_id, sess_id, strategy_filter="mb", num_tents=12)
encoder_mb.fit_encoder()

encoder_mf = Encoder(subj_id, sess_id, strategy_filter="mf", num_tents=12)
encoder_mf.fit_encoder()

In [ ]:
encoder_mf.verify()

In [ ]:
encoder.trial_data["current_block_side"]

In [ ]:
# weight comparison
import numpy as np

regr = "response"
val = "left"

i = np.where(encoder.dm_names == f"{regr}_{val}")[0][0]

plt.figure(tight_layout=True)
plt.scatter(
    encoder_mb.encoder.coef_[:, i], encoder_mf.encoder.coef_[:, i], s=0.5, alpha=0.5
)

plt.axhline(y=0, color="k")
plt.axvline(x=0, color="k")
plt.xlabel(f"mb, bweight {regr} {val}")
plt.ylabel(f"mf, bweight {regr} {val}")
plt.plot()

In [ ]:
# TODO:
# --DONE--
# make it easier to access the dm index of a regressor by name
# --------